
# 🏗 Core Vision Perfect V1 — 01 · Build Artifacts (Colab)

Builds **everything the search system needs** from the organiser dataset on
your Drive: catalog → keyframe self-extraction (K-batches) → dense embeddings
→ FAISS indexes → OCR / ASR / captions → persisted BM25 text index.

**Every step is resumable** — if Colab disconnects, just *Runtime → Run all*
again; finished videos are skipped. Stage timings land in
`artifacts/logs/nb01.log`.

Prerequisites (see `docs/DRIVE_SETUP.md`):
`MyDrive/AIC2025/data/{keyframes, map-keyframes, media-info, clip-features-32, objects, videos}`

GPU: any (T4 works; A100/H100 much faster for SigLIP-2 + captions).


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  1 · PARAMS — the ONLY cell you may need to edit                 ║
# ╚══════════════════════════════════════════════════════════════════╝
DRIVE_PROJECT_DIR = "AIC2025"        # MyDrive/<this>/{data, artifacts}
REPO_URL  = "https://github.com/ledinhminhquan/Core-Vision_Perfect_V1.git"
REPO_REF  = "main"

# Which dense encoders to build indexes for (order = ensemble order).
#   "siglip2"          multilingual default (needed for training too)
#   "openclip"         English lane (DFN5B ViT-H/14-378) — strongest with translation
#   "qwen_embed"       optional HEAVY lane (Qwen embedding tower — strong, slow)
#   "provided_clip32"  organiser features — instant, no GPU (L-batches only);
#                      auto-added in the catalog cell when clip-features-32 exists
EMBED_MODELS = ["siglip2", "openclip"]

# Copy keyframes from Drive → local disk before embedding (much faster I/O).
COPY_KEYFRAMES_LOCAL = True

# Aux indexes to build (each is resumable; captions are the slowest).
RUN_OCR, RUN_ASR, RUN_CAPTIONS = True, True, True
CAPTION_STRIDE = 2                   # caption every 2nd keyframe (2× faster)

# K-batch shot detection: install TransNetV2 (the winning-team detector) for
# keyframe self-extraction. Installed --no-deps (Colab torch is never touched);
# without it extraction falls back to PySceneDetect automatically. (nb01 only)
INSTALL_TRANSNETV2 = True

# Force-rebuild toggles — mặc định False = resume/skip khi artifact đã có.
FORCE_CATALOG    = False             # rebuild the catalog parquet
FORCE_EMBED      = False             # re-embed every keyframe
FORCE_INDEX      = False             # rebuild the FAISS indexes
FORCE_AUX        = False             # redo OCR/ASR/captions from scratch
FORCE_TEXT_INDEX = False             # rebuild the persisted BM25 text index

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("params ok")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2 · Mount Drive + folder layout + preflight write test          ║
# ╚══════════════════════════════════════════════════════════════════╝
import os, time
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")
assert Path("/content/drive/MyDrive").exists(), "Drive mount failed — rerun this cell"

PROJECT   = Path("/content/drive/MyDrive") / DRIVE_PROJECT_DIR
DATA_DIR  = PROJECT / "data"           # organiser dataset (merged packages)
ARTIFACTS = PROJECT / "artifacts"      # everything we build → survives disconnects
for p in (DATA_DIR, ARTIFACTS):
    p.mkdir(parents=True, exist_ok=True)

# PREFLIGHT (v12): Drive PHẢI ghi/đọc được — quota đầy hay mất quyền thì
# dừng NGAY tại đây thay vì hỏng giữa chừng sau 2 giờ chạy.
_probe = ARTIFACTS / f"_write_test_{int(time.time())}.tmp"
try:
    _probe.write_text("ok", encoding="utf-8")
    assert _probe.read_text(encoding="utf-8") == "ok"
    _probe.unlink()
    print("✅ Drive write test: OK")
except Exception as e:
    raise RuntimeError(
        f"❌ Không ghi được vào Drive ({ARTIFACTS}): {e!r}\n"
        "Kiểm tra dung lượng (quota) Google Drive và quyền truy cập thư mục, "
        "rồi chạy lại ô này."
    ) from e

# HF + pip caches on Drive → models/wheels download once, not per session.
os.environ["HF_HOME"] = str(ARTIFACTS / "hf_cache")
os.environ["PIP_CACHE_DIR"] = str(ARTIFACTS / "pip_cache")
for _d in (os.environ["HF_HOME"], os.environ["PIP_CACHE_DIR"]):
    Path(_d).mkdir(parents=True, exist_ok=True)

import shutil
free_gb = shutil.disk_usage(str(PROJECT)).free / 1e9
print(f"Project: {PROJECT}")
print(f"Drive free space: {free_gb:.0f} GB")
if free_gb < 20:
    print("⚠ Less than 20 GB free on Drive — embeddings/checkpoints may not fit!")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  3 · Get repo + install dependencies (v12 discipline)            ║
# ╚══════════════════════════════════════════════════════════════════╝
# Quy tắc (học từ notebook Toxicity v12):
#   * check version qua importlib.metadata — KHÔNG import package trước khi
#     nâng cấp (import sớm sẽ ghim version cũ vào sys.modules);
#   * KHÔNG BAO GIỜ đụng torch/torchvision/torchaudio của Colab;
#   * chỉ cài đúng những gói thiếu/sai version (--prefer-binary);
#   * sau khi cài: `pip check` + micro-fix (tối đa 2 vòng, không crash),
#     rồi purge sys.modules TRƯỚC khi import cvp.
FORCE_REINSTALL_DEPS = False

import re, subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/Core-Vision_Perfect_V1")

def _run(cmd, show=None, **kw):
    # `show` masks credentials in the echoed command — a PAT-carrying clone
    # URL must NEVER be printed into the saved notebook output.
    print("$", " ".join(map(str, show or cmd)))
    return subprocess.run([str(c) for c in cmd], check=False, **kw).returncode

def _pip(args):
    return _run([sys.executable, "-m", "pip", *args])

# Private repo? Add a fine-grained PAT as Colab secret "GITHUB_TOKEN"
# (Contents: Read-only on this repo) and ENABLE its notebook-access toggle.
clone_url, _tok = REPO_URL, None
try:
    from google.colab import userdata
    _tok = userdata.get("GITHUB_TOKEN")
except Exception as _e:
    print(f"⚠ KHÔNG đọc được secret GITHUB_TOKEN ({type(_e).__name__}) — repo "
          "private sẽ KHÔNG clone được. Kiểm tra: 🔑 panel có secret tên đúng "
          "y hệt GITHUB_TOKEN và công tắc 'Notebook access' đã BẬT chưa?")
if _tok and clone_url.startswith("https://github.com/"):
    clone_url = clone_url.replace("https://", f"https://{_tok}@")
    print(f"GITHUB_TOKEN: loaded ({len(_tok)} chars, {_tok[:11]}…)")
elif not _tok:
    print("⚠ GITHUB_TOKEN trống/vắng mặt — thử clone KHÔNG xác thực "
          "(chắc chắn fail nếu repo private).")

if REPO_DIR.exists():
    _run(["git", "-C", REPO_DIR, "fetch", "--all", "-q"])
    _run(["git", "-C", REPO_DIR, "checkout", REPO_REF, "-q"])
    _run(["git", "-C", REPO_DIR, "pull", "-q"])
else:
    rc = _run(["git", "clone", "--branch", REPO_REF, clone_url, REPO_DIR],
              show=["git", "clone", "--branch", REPO_REF, REPO_URL, REPO_DIR])
    if rc != 0:  # private repo / no network → fall back to a Drive copy
        print("⚠ Clone THẤT BẠI. Nguyên nhân thường gặp, theo thứ tự:\n"
              "  1) Secret GITHUB_TOKEN sai tên / chưa bật Notebook access "
              "(xem cảnh báo phía trên);\n"
              "  2) PAT sai/hết hạn/thiếu quyền — cần fine-grained PAT với "
              "Contents: Read-only cấp cho ĐÚNG repo này;\n"
              "  3) Mạng Colab trục trặc tạm thời — chạy lại cell.")
        drive_copy = Path("/content/drive/MyDrive") / DRIVE_PROJECT_DIR / "Core-Vision_Perfect_V1"
        assert drive_copy.exists(), (
            "Clone failed and no Drive copy found. Either make the GitHub repo "
            f"reachable or upload the repo folder to {drive_copy}"
        )
        import shutil as _sh
        _sh.copytree(drive_copy, REPO_DIR)
        print("Using repo copy from Drive")

try:
    from packaging.requirements import Requirement
except ImportError:
    _pip(["install", "-q", "packaging"])
    from packaging.requirements import Requirement
from importlib.metadata import PackageNotFoundError
from importlib.metadata import version as _meta_version

# Parse requirements-colab.txt; strip any torch* line (Colab rule #1: the
# preinstalled torch/torchvision/torchaudio build must never be touched).
reqs = []
for _line in (REPO_DIR / "requirements-colab.txt").read_text(encoding="utf-8").splitlines():
    _line = _line.split("#", 1)[0].strip()
    if not _line:
        continue
    try:
        _r = Requirement(_line)
    except Exception:
        print("⚠ bỏ qua requirement không parse được:", _line)
        continue
    if _r.name.lower().replace("-", "_").startswith("torch"):
        print("skip (never touch Colab torch):", _line)
        continue
    reqs.append(_r)

def _satisfied(r):
    """Installed + in range — via importlib.metadata, WITHOUT importing it."""
    try:
        v = _meta_version(r.name)
    except PackageNotFoundError:
        return False
    return (not r.specifier) or r.specifier.contains(v, prereleases=True)

missing = [r for r in reqs if FORCE_REINSTALL_DEPS or not _satisfied(r)]
did_install = bool(missing)
if missing:
    print(f"installing {len(missing)} package(s):", ", ".join(r.name for r in missing))
    _pip(["install", "-q", "--prefer-binary", *[str(r) for r in missing]])
else:
    print("dependencies satisfied — no pip install needed")

_pip(["install", "-q", "-e", str(REPO_DIR), "--no-deps"])

# faiss: gpu wheel with cpu fallback (metadata check — no import)
def _installed(*names):
    for n in names:
        try:
            _meta_version(n)
            return n
        except PackageNotFoundError:
            pass
    return None

if _installed("faiss-gpu-cu12", "faiss-gpu", "faiss-cpu", "faiss") is None:
    if _pip(["install", "-q", "faiss-gpu-cu12"]) != 0:
        _pip(["install", "-q", "faiss-cpu"])
    did_install = True

# `pip check` + micro-fixes for known conflicts (max 2 rounds, then warn)
def _pip_check():
    r = subprocess.run([sys.executable, "-m", "pip", "check"],
                       capture_output=True, text=True)
    return r.returncode, ((r.stdout or "") + "\n" + (r.stderr or "")).strip()

if did_install:
    rc, out = _pip_check()
    for _round in (1, 2):
        if rc == 0:
            break
        # pip's two REAL formats (round-3 fix L-R3-8 — the old regex missed the
        # version-conflict wording so that repair branch never ran):
        #   "pkgA 1.0 requires pkgB, which is not installed."
        #   "pkgA 1.0 has requirement pkgB<2,>=1, but you have pkgB 3.0."
        _specs = sorted({
            m.strip()
            for m in re.findall(
                r"(?:requires|has requirement) (.+?), (?:but you have|which is not installed)", out)
            if not m.strip().lower().startswith("torch")
        })
        if not _specs:
            break
        print(f"pip check micro-fix (round {_round}):", ", ".join(_specs))
        _pip(["install", "-q", "--prefer-binary", *_specs])
        rc, out = _pip_check()
    print("pip check: OK" if rc == 0 else f"⚠ pip check còn cảnh báo (không chặn):\n{out}")

# Purge stale sys.modules of upgraded packages BEFORE importing cvp (v12).
# ONLY the packages actually (re)installed THIS run (round-11): purging every
# requirement dropped numpy/pandas from sys.modules while torch still held
# references to the old modules — the "NumPy module was reloaded" warning.
if did_install:
    _ALIAS = {"pillow": "pil", "pyyaml": "yaml", "opencv_python_headless": "cv2",
              "open_clip_torch": "open_clip", "scikit_learn": "sklearn"}
    _roots = {r.name.lower().replace("-", "_") for r in missing} | {"cvp", "faiss"}
    _roots |= {_ALIAS[n] for n in _roots & set(_ALIAS)}
    _purged = [m for m in list(sys.modules)
               if m.split(".", 1)[0].lower().replace("-", "_") in _roots]
    for _m in _purged:
        sys.modules.pop(_m, None)
    if _purged:
        print(f"purged {len(_purged)} stale sys.modules entries")

    # Sanity (round-11): the HF stack must import cleanly in a FRESH
    # interpreter — a broken hub/accelerate pairing must surface HERE with an
    # actionable message, not 5 cells later as a cryptic circular import.
    _rc = _run([sys.executable, "-c", "import transformers, accelerate"])
    if _rc != 0:
        print("⚠ transformers/accelerate KHÔNG import được — thường do phiên cài "
              "này đã hạ cấp huggingface-hub dưới mức accelerate cần. Cách sửa "
              "sạch nhất: Runtime ▸ Disconnect and delete runtime, rồi Run all "
              "lại từ đầu (mọi tiến độ đã nằm trên Drive, không mất gì).")

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
import cvp
print("cvp", cvp.__version__, "ready")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  4 · Point cvp at the data + GPU setup (TF32 / SDPA / bf16)      ║
# ╚══════════════════════════════════════════════════════════════════╝
import os, torch

os.environ["CVP_PATHS__DATA_ROOT"]      = str(DATA_DIR)
os.environ["CVP_PATHS__ARTIFACTS_ROOT"] = str(ARTIFACTS)
os.environ["CVP_SETTINGS"] = str(REPO_DIR / "configs" / "settings.yaml")

print("torch", torch.__version__, "| CUDA build", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())
GPU_NAME, VRAM_GB, USE_BF16 = "cpu", 0.0, False
if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    USE_BF16 = torch.cuda.is_bf16_supported()
    # TF32 fast paths (new API with old fallback)
    try:
        torch.backends.cuda.matmul.fp32_precision = "tf32"
        torch.backends.cudnn.conv.fp32_precision = "tf32"
    except Exception:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    for fn in ("enable_flash_sdp", "enable_mem_efficient_sdp"):
        if hasattr(torch.backends.cuda, fn):
            getattr(torch.backends.cuda, fn)(True)
print(f"GPU: {GPU_NAME} | VRAM {VRAM_GB:.0f} GB | bf16={USE_BF16}")

# Colab secrets → env (optional: Gemini query enhancement/VQA, HF pushes).
# GOOGLE_API_KEY is the name Colab's built-in "Gemini API key ▸ Import from
# Google AI Studio" button creates — the engine accepts either spelling.
try:
    from google.colab import userdata
    for _sec in ("GEMINI_API_KEY", "GOOGLE_API_KEY", "HF_TOKEN"):
        try:
            _v = userdata.get(_sec)
            if _v:
                os.environ[_sec] = _v
                print(f"secret {_sec}: loaded")
        except Exception:
            pass
except ImportError:
    pass

from cvp.config import load_settings
from cvp.utils.logging import setup_logging
settings = load_settings()
setup_logging("INFO")
print("data_root      =", settings.paths.data_root)
print("artifacts_root =", settings.paths.artifacts_root)

In [ ]:
# ── 5 · (optional) Auto-extract organiser zips still sitting in data/ ──
# If you uploaded raw zips (Keyframes_L21.zip, ...) instead of extracted
# folders, this unpacks them into the right places, then removes nothing.
import zipfile
from pathlib import Path

ZIP_DEST = {
    "keyframes":       DATA_DIR / "keyframes",
    "videos":          DATA_DIR / "videos",
    "clip-features":   DATA_DIR / "clip-features-32",
    "map-keyframes":   DATA_DIR / "map-keyframes",
    "media-info":      DATA_DIR / "media-info",
    "objects":         DATA_DIR / "objects",
}

def guess_dest(zname: str):
    # Normalise _/space → '-' so 2026 spelling variants (clip_features,
    # Map Keyframes, keyframe singular, video_…) still route correctly.
    # SPECIFIC families are tested FIRST (review R3-C25): a 2026 rename like
    # "keyframe-map-b1.zip" must never fall into the generic keyframes bucket.
    z = zname.lower().replace("_", "-").replace(" ", "-")
    if "map" in z and "keyframe" in z:                        return ZIP_DEST["map-keyframes"]
    if "clip-feature" in z or "features-32" in z:             return ZIP_DEST["clip-features"]
    if "media-info" in z or "metadata" in z:                  return ZIP_DEST["media-info"]
    if "object" in z:                                         return ZIP_DEST["objects"]
    if z.startswith(("keyframes", "keyframe", "key-frames")): return ZIP_DEST["keyframes"]
    if z.startswith(("videos", "video-")):                    return ZIP_DEST["videos"]
    return None

import re as _re
import shutil as _sh
_VID_DIR_RE = _re.compile(r"^[A-Z]\d{2}_V\d{3}$")   # per-video PAYLOAD dirs, never wrappers

_unknown_zips = []
for zp in sorted(DATA_DIR.glob("*.zip")):
    dest = guess_dest(zp.name)
    if dest is None:
        _unknown_zips.append(zp.name); continue
    marker = dest / f".unzipped-{zp.stem}"
    if marker.exists():
        continue
    print("unzipping", zp.name, "→", dest)
    dest.mkdir(parents=True, exist_ok=True)
    tmp_root = dest.parent / "__tmp_unzip"
    if tmp_root.exists():
        _sh.rmtree(tmp_root)
    with zipfile.ZipFile(zp) as z:
        z.extractall(tmp_root)
    # Walk down through GENUINE wrapper folders only: a single child dir that
    # is NOT video-id-shaped. Handles nested shapes like Videos_L28_a/video/*
    # or Keyframes_L26/keyframes/L26_*/ (review R3-C28) while never mistaking
    # a lone per-video payload dir for a wrapper (review R3-C14).
    src = tmp_root
    while True:
        children = list(src.iterdir())
        if len(children) == 1 and children[0].is_dir() and not _VID_DIR_RE.match(children[0].name):
            src = children[0]
            continue
        break
    # MERGE into dest — never overwrite, never drop a second zip's files.
    kept_existing = 0
    for item in src.iterdir():
        target = dest / item.name
        if not target.exists():
            _sh.move(str(item), str(target))
        elif item.is_dir() and target.is_dir():
            for sub in item.iterdir():
                sub_target = target / sub.name
                if not sub_target.exists():
                    _sh.move(str(sub), str(sub_target))
                else:
                    kept_existing += 1
        else:
            kept_existing += 1
    _sh.rmtree(tmp_root, ignore_errors=True)
    if kept_existing:
        print(f"   giữ nguyên {kept_existing} mục đã tồn tại (không ghi đè)")
    marker.touch()
if _unknown_zips:
    print("\n" + "!" * 70)
    print("⚠ CÁC ZIP KHÔNG NHẬN DIỆN ĐƯỢC (KHÔNG được giải nén — kiểm tra tay!):")
    for _n in _unknown_zips:
        print("   •", _n)
    print("  Nếu đây là gói 2026 với tên mới → giải nén thủ công vào đúng thư mục")
    print("  data/{keyframes,map-keyframes,media-info,clip-features-32,objects,videos}")
    print("  (mapping: docs/DATASET_INGESTION.md §2) rồi chạy lại từ ô này.")
    print("!" * 70)
print("zip check done")

In [ ]:
# ── 6 · Copy keyframes Drive → local disk (I/O speed) ──
# Embedding reads hundreds of thousands of small JPGs; Drive FUSE is ~50×
# slower than local disk. Artifacts still go to Drive.
import shutil
from pathlib import Path

if COPY_KEYFRAMES_LOCAL and (DATA_DIR / "keyframes").exists():
    LOCAL_DATA = Path("/content/data")
    (LOCAL_DATA).mkdir(exist_ok=True)
    for sub in ("keyframes", "map-keyframes", "media-info", "objects", "clip-features-32"):
        src, dst = DATA_DIR / sub, LOCAL_DATA / sub
        if not src.exists():
            continue
        if not dst.exists():
            print(f"copying {sub} → local ...")
            # copy into a tmp dir then rename: an interrupted copy must not
            # leave a partial folder that a re-run would silently accept
            tmp_dst = LOCAL_DATA / (sub + ".__tmp")
            if tmp_dst.exists():
                shutil.rmtree(tmp_dst)
            shutil.copytree(src, tmp_dst)
            tmp_dst.rename(dst)
            continue
        # Local dir already exists: MERGE any children Drive has that local
        # lacks — a batch added mid-session (unzip cell re-run, manual upload)
        # must reach local instead of being silently skipped (review R3-C15).
        # Same tmp+rename discipline as the first copy: an interrupted merge
        # must not leave a partial video dir that the next run would accept
        # and the catalog would silently index half-empty (review R4).
        for stale in dst.glob("*.__tmp"):
            shutil.rmtree(stale, ignore_errors=True) if stale.is_dir() else stale.unlink()
        added = 0
        for item in src.iterdir():
            target = dst / item.name
            if target.exists():
                continue
            tmp_target = dst / (item.name + ".__tmp")
            if item.is_dir():
                shutil.copytree(item, tmp_target)
            else:
                shutil.copy2(item, tmp_target)
            tmp_target.rename(target)
            added += 1
        if added:
            print(f"merged {added} new item(s) from Drive into local {sub}/")
    # videos stay on Drive (huge); link them in
    if (DATA_DIR / "videos").exists() and not (LOCAL_DATA / "videos").exists():
        (LOCAL_DATA / "videos").symlink_to(DATA_DIR / "videos")
    import os
    os.environ["CVP_PATHS__DATA_ROOT"] = str(LOCAL_DATA)
    from cvp.config import load_settings
    settings = load_settings()
    print("data_root now:", settings.paths.data_root)
else:
    print("using Drive data_root directly")

In [ ]:
# ── 7 · Catalog + self-extract K-batch keyframes (+ sync back to Drive) ──
import time
from contextlib import contextmanager

@contextmanager
def _log_stage(name: str):
    """Tee nhẹ kiểu v12: ghi start/end/duration của mỗi stage lên Drive."""
    log_path = ARTIFACTS / "logs" / "nb01.log"
    log_path.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')} START {name}\n")
    try:
        yield
    finally:
        with open(log_path, "a", encoding="utf-8") as f:
            f.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')} END   {name} "
                    f"({time.time() - t0:.0f}s)\n")

from cvp.data.catalog import KeyframeCatalog
from cvp.data.extraction import extract_missing

# TransNetV2 for K-batch shot detection (best detector per the winning teams).
# --no-deps: torch/numpy/opencv already exist — Colab torch must not be touched.
# ffmpeg-python is REQUIRED by predict_video() (round-3 fix M-R3-1) and itself
# hard-imports `past.builtins` from the `future` distribution at import time
# (round-4 fix: `future` is a REAL runtime dep, not a py2 leftover). All three
# packages DO declare dependencies, but every one of them is either already on
# Colab or installed by this very call — so --no-deps stays torch-safe (round-5
# wording fix C-R5-2; the ffmpeg BINARY ships with Colab).
# Missing/failed install is fine: extraction falls back to PySceneDetect.
if INSTALL_TRANSNETV2:
    def _transnet_ready() -> bool:
        try:
            import transnetv2_pytorch  # noqa: F401
            import ffmpeg  # noqa: F401
            return True
        except ImportError:
            return False

    if _transnet_ready():
        print("TransNetV2 + ffmpeg-python: already installed")
    else:
        _pip(["install", "-q", "--no-deps", "transnetv2-pytorch", "ffmpeg-python", "future"])
        # Verify-then-report: a green pip rc alone proved nothing in round 3.
        print("TransNetV2:", "READY" if _transnet_ready()
              else "unavailable — SceneDetect fallback will be used")

with _log_stage("catalog"):
    n_extracted = extract_missing(settings)     # K-videos without keyframes
    print("extracted videos:", n_extracted)
    catalog = KeyframeCatalog(settings)
    df = catalog.build(force=FORCE_CATALOG or n_extracted > 0)
print(f"catalog: {len(df):,} keyframes / {df.video_id.nunique()} videos "
      f"({int(df.has_map.sum()):,} frames with map-keyframes)")
_no_map = int((~df.has_map).sum())
if _no_map:
    print(f"⚠ {_no_map:,} keyframes KHÔNG có map-keyframes → frame_idx đang là "
          "ƯỚC LƯỢNG, nộp bài sẽ SAI. Upload gói map-keyframes của BTC, hoặc "
          "tái dựng từ video gốc: python scripts/05_rebuild_map_keyframes.py "
          "(cần videos/*.mp4), rồi chạy lại ô này với FORCE_CATALOG=True.")

# K-batch sync-back: khi COPY_KEYFRAMES_LOCAL=True, extract_missing ghi
# keyframes + map-keyframes mới vào data_root LOCAL (/content/data) — local
# disk BỐC HƠI khi hết session, NB3/laptop sẽ không bao giờ thấy chúng.
# Mirror mọi folder/CSV mà Drive CHƯA có về DATA_DIR (tmp + rename như ô 6).
# No-op khi chạy thẳng trên Drive hoặc không có gì mới.
from pathlib import Path
import shutil as _sh

n_synced = 0
_local_root = Path(str(settings.paths.data_root)).resolve()
if _local_root != DATA_DIR.resolve():
    for src_root, dst_root, want_dir in (
        (_local_root / "keyframes", DATA_DIR / "keyframes", True),
        (_local_root / "map-keyframes", DATA_DIR / "map-keyframes", False),
    ):
        if not src_root.is_dir():
            continue
        for item in sorted(src_root.iterdir()):
            if item.name.endswith(".__tmp"):
                continue
            if not (item.is_dir() if want_dir else item.suffix == ".csv"):
                continue
            target = dst_root / item.name
            if target.exists():
                continue
            dst_root.mkdir(parents=True, exist_ok=True)
            tmp = dst_root / (item.name + ".__tmp")
            if tmp.is_dir():
                _sh.rmtree(tmp)
            elif tmp.exists():
                tmp.unlink()
            (_sh.copytree if want_dir else _sh.copy2)(item, tmp)
            tmp.rename(target)
            n_synced += 1
print(f"sync-back to Drive: {n_synced} item(s)" if n_synced
      else "sync-back: nothing new for Drive")

# Organiser CLIP features → free extra retrieval lane, but ONLY with FULL
# coverage: ingest_provided_features bỏ qua video không có .npy, rồi
# store.build sẽ hard-fail trên lane thiếu vector (video K-batch tự extract
# KHÔNG BAO GIỜ có organiser features).
feat_dir = settings.paths.data(settings.paths.clip_features_dir)
if feat_dir.is_dir() and "provided_clip32" not in EMBED_MODELS:
    _vids = set(map(str, df.video_id.unique()))
    _have = {p.stem for p in feat_dir.glob("*.npy")}
    _missing_feats = sorted(_vids - _have)
    if not _missing_feats:
        EMBED_MODELS = EMBED_MODELS + ["provided_clip32"]
        print("auto-added 'provided_clip32' to EMBED_MODELS (full .npy coverage)")
    else:
        print(f"provided_clip32 lane skipped: {len(_missing_feats)} video(s) have no "
              f"organiser features (K-batch present?) — e.g. {_missing_feats[:3]}")

In [ ]:
# ── 8 · Dense embeddings + FAISS index per model (resumable) ──
# ⚠ model_tag được in TRƯỚC khi embed từng lane. Nếu embedder dừng với lỗi
# "model tag mismatch": session này nạp CHECKPOINT KHÁC session trước (vd.
# openclip: hub PE-Core không tải được → fallback DFN5B) — trộn 2 checkpoint
# trong một folder embeddings sẽ hỏng index. Chạy lại với FORCE_EMBED=True
# để re-embed sạch, hoặc khôi phục mạng để nạp đúng checkpoint cũ.
import gc, torch
from cvp.index.embedder import embed_all_keyframes, ingest_provided_features
from cvp.index.store import IndexStore
from cvp.models.registry import build_model

for name in EMBED_MODELS:
    print(f"\n════ {name} ════")
    model_tag = None
    with _log_stage(f"embed:{name}"):
        if name == "provided_clip32":
            ingest_provided_features(settings, catalog)
        else:
            model = build_model(settings, name)
            # the EXACT loaded checkpoint, printed up front so a cross-session
            # mismatch is diagnosable; also stamps the index (scripts/02 contract)
            model_tag = getattr(model, "model_tag", None) or model.key
            print(f"[{name}] model_tag = {model_tag}")
            embed_all_keyframes(model, settings, catalog, overwrite=FORCE_EMBED)
            del model; gc.collect(); torch.cuda.empty_cache()
        store = IndexStore(settings, name)
        store.build(catalog, force=FORCE_INDEX or FORCE_EMBED, model_tag=model_tag)
    print(f"[{name}] index: {store.count():,} vectors, dim {store.dim()}")

In [ ]:
# ── 9 · Aux indexes: OCR / ASR / captions (each resumable) ──
# AUX_CHANGED feeds cell 10: scripts/03's rule is "force-rebuild the BM25
# text index whenever this run processed ≥1 aux video".
AUX_CHANGED = False

def _did_work(n) -> bool:
    return n is None or n > 0   # None (older API) → assume something changed

if RUN_OCR:
    from cvp.auxindex.ocr import ocr_all_keyframes
    with _log_stage("ocr"):
        n = ocr_all_keyframes(settings, catalog, overwrite=FORCE_AUX)
    AUX_CHANGED = AUX_CHANGED or _did_work(n)
    print(f"OCR: processed {n} videos")
if RUN_ASR:
    from cvp.auxindex.asr import asr_all_videos
    with _log_stage("asr"):
        n = asr_all_videos(settings, catalog, overwrite=FORCE_AUX)
    AUX_CHANGED = AUX_CHANGED or _did_work(n)
    print(f"ASR: processed {n} videos")
if RUN_CAPTIONS:
    from cvp.auxindex.captioner import caption_all_keyframes
    with _log_stage("captions"):
        n = caption_all_keyframes(settings, catalog, stride=CAPTION_STRIDE, overwrite=FORCE_AUX)
    AUX_CHANGED = AUX_CHANGED or _did_work(n)
    print(f"Captions: processed {n} videos")
print("aux indexes done | AUX_CHANGED =", AUX_CHANGED)

In [ ]:
# ── 10 · Persisted BM25 text index (OCR/ASR/captions/metadata) ──
# Query-time BM25 becomes candidate-restricted lookups instead of rescanning
# every artifact on engine start. Direct function call into scripts/03 (the
# module name starts with a digit → import_module, not a plain import).
import sys
from importlib import import_module

if str(REPO_DIR / "scripts") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "scripts"))
build_text_index = import_module("03_build_aux_indexes").build_text_index

# scripts/03 rule: rebuild whenever the aux cell processed ≥1 video this run
# (AUX_CHANGED), not only when the catalog signature changed.
with _log_stage("text_index"):
    built = build_text_index(settings, catalog, force=FORCE_TEXT_INDEX or AUX_CHANGED)
if built:
    print("text index built: [" + ", ".join(built) + "]")
else:
    print("text index up-to-date (signature match) — đặt FORCE_TEXT_INDEX=True để build lại")

In [ ]:
# ── 11 · Health report ──
import json
from cvp.pipeline.ingest import doctor
print(json.dumps(doctor(settings), indent=2, ensure_ascii=False))
print("\n✅ Artifacts build complete. Next: notebooks/02_train_vi_encoder_H100.ipynb")